In [ ]:
%%capture
!pip install pip3-autoremove
!pip-autoremove torch torchvision torchaudio -y
!pip install torch torchvision torchaudio xformers --index-url https://download.pytorch.org/whl/cu121
!pip install --no-deps bitsandbytes accelerate xformers==0.0.29.post3 peft trl triton cut_cross_entropy unsloth_zoo
!pip install sentencepiece protobuf datasets huggingface_hub hf_transfer
!pip install --no-deps git+https://github.com/rupaut98/unsloth.git@fix-cpt-error

In [ ]:
# Add after imports but before model loading
import torch.amp.grad_scaler
from unsloth import is_bfloat16_supported

# Only patch if needed (FP16-only hardware)
def patch_grad_scaler_if_needed(model=None, target_modules=None):
    """Conditionally patch PyTorch's GradScaler based on hardware and model configuration"""
    # Check if we're on hardware without BF16 support
    if not is_bfloat16_supported():
        # Skip patching for Gemma-3 models
        is_gemma3 = False
        if model is not None:
            # Check model name or configuration for "gemma-3"
            model_name = getattr(model, "name_or_path", "")
            if not model_name and hasattr(model, "config"):
                model_name = getattr(model.config, "name_or_path", "")
                if not model_name and hasattr(model.config, "_name_or_path"):
                    model_name = model.config._name_or_path
            is_gemma3 = "gemma-3" in str(model_name).lower()

        if is_gemma3:
            print("Unsloth: Detected Gemma-3 model, skipping GradScaler patch")
            return False

        # Check if we're training embedding layers (either from arguments or manually check)
        train_embeddings = False
        if target_modules is not None:
            train_embeddings = "embed_tokens" in target_modules or "lm_head" in target_modules
        elif model is not None:
            # Look through model parameters for embedding layers
            for name, _ in model.named_parameters():
                if "embed_tokens" in name or "lm_head" in name:
                    train_embeddings = True
                    break

        if train_embeddings:
            # Only patch if we're training embedding layers on FP16-only hardware
            original_unscale_grads = torch.amp.grad_scaler.GradScaler._unscale_grads_

            def patched_unscale_grads(self, optimizer, inv_scale, found_inf, allow_fp16=False):
                return original_unscale_grads(self, optimizer, inv_scale, found_inf, True)

            # Apply the patch
            torch.amp.grad_scaler.GradScaler._unscale_grads_ = patched_unscale_grads
            print("Unsloth: Patched GradScaler to allow FP16 gradients for embedding training")
            return True

    return False

# Call the function with your target modules before model creation
target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                 "gate_proj", "up_proj", "down_proj",
                 "embed_tokens", "lm_head"]

In [ ]:
from unsloth import FastLanguageModel
import torch
max_seq_length = 2048 # Choose any! We auto support RoPE Scaling internally!
dtype = None # None for auto detection. Float16 for Tesla T4, V100, Bfloat16 for Ampere+
load_in_4bit = True # Use 4bit quantization to reduce memory usage. Can be False.

# 4bit pre quantized models we support for 4x faster downloading + no OOMs.
fourbit_models = [
    "unsloth/mistral-7b-v0.3-bnb-4bit",      # New Mistral v3 2x faster!
    "unsloth/mistral-7b-instruct-v0.3-bnb-4bit",
    "unsloth/llama-3-8b-bnb-4bit",           # Llama-3 15 trillion tokens model 2x faster!
    "unsloth/llama-3-8b-Instruct-bnb-4bit",
    "unsloth/llama-3-70b-bnb-4bit",
    "unsloth/Phi-3-mini-4k-instruct",        # Phi-3 2x faster!
    "unsloth/Phi-3-medium-4k-instruct",
    "unsloth/mistral-7b-bnb-4bit",
    "unsloth/gemma-7b-bnb-4bit",             # Gemma 2.2x faster!
] # More models at https://huggingface.co/unsloth

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Qwen2.5-Math-1.5B-bnb-4bit", # Choose ANY! eg teknium/OpenHermes-2.5-Mistral-7B
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
    # token = "hf_...", # use one if using gated models like meta-llama/Llama-2-7b-hf
)

In [ ]:
# Apply conditional patch
patch_applied = patch_grad_scaler_if_needed(model=model, target_modules=target_modules)

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r = 64, # Choose any number > 0 ! Suggested 8, 16, 32, 64, 128
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj",

                      "embed_tokens", "lm_head",], # Add for continual pretraining
    lora_alpha = 64,
    lora_dropout = 0, # Supports any, but = 0 is optimized
    bias = "none",    # Supports any, but = "none" is optimized
    # [NEW] "unsloth" uses 30% less VRAM, fits 2x larger batch sizes!
    use_gradient_checkpointing = "unsloth", # True or "unsloth" for very long context
    random_state = 3407,
    use_rslora = True,   # We support rank stabilized LoRA
    loftq_config = None, # And LoftQ
)

In [ ]:
from datasets import load_dataset
import os # Import os to check file existence

book_file_path = "/kaggle/input/book-batch1/cleaned_book_batch1.md"
paper_file_path = "/kaggle/input/paper-batch/cleaned_batch1.md" # Corrected path assuming this is intended
val_file_path = "/kaggle/input/val-data/cleaned_batch2.md"
val_lines_to_load = 8000

# --- Check if files exist ---
print(f"Checking existence of: {book_file_path}, {paper_file_path}, {val_file_path}")
missing_files = []
if not os.path.exists(book_file_path):
    missing_files.append(book_file_path)
if not os.path.exists(paper_file_path):
    missing_files.append(paper_file_path)
if not os.path.exists(val_file_path):
    missing_files.append(val_file_path)

if missing_files:
    print(f"Error: The following data files were not found:")
    for f in missing_files:
        print(f"- {f}")
    # Optional: List directories for debugging
    print("\nListing contents of /kaggle/input/ for context:")
    try:
        print(os.listdir("/kaggle/input/"))
        if os.path.exists(os.path.dirname(book_file_path)): print(f"Contents of {os.path.dirname(book_file_path)}: {os.listdir(os.path.dirname(book_file_path))}")
        if os.path.exists(os.path.dirname(paper_file_path)): print(f"Contents of {os.path.dirname(paper_file_path)}: {os.listdir(os.path.dirname(paper_file_path))}")
        if os.path.exists(os.path.dirname(val_file_path)): print(f"Contents of {os.path.dirname(val_file_path)}: {os.listdir(os.path.dirname(val_file_path))}")
    except Exception as e:
        print(f"Could not list input directory contents: {e}")
    raise FileNotFoundError(f"Required data files not found: {', '.join(missing_files)}")
else:
    print("All required data files found.")
# --- End Check ---


# --- Load and Combine Training Data ---
print("Loading and combining training data...")
book_text = ""
with open(book_file_path, 'r', encoding='utf-8') as f:
    book_text = f.read()

paper_text = ""
with open(paper_file_path, 'r', encoding='utf-8') as f:
    paper_text = f.read()

# Combine book first, then paper
combined_train_text = book_text + "\n" + paper_text # Add newline separator

# Create combined training dataset object
train_dataset_raw = Dataset.from_dict({"text": [combined_train_text]})
print("Raw Training Dataset created:")
print(train_dataset_raw)


# --- Load Limited Validation Data ---
print(f"Loading {val_lines_to_load} lines of validation data...")
val_lines = []
with open(val_file_path, 'r', encoding='utf-8') as f:
    val_lines = f.readlines()[:val_lines_to_load]
val_text = "".join(val_lines)

# Create validation dataset object
eval_dataset_raw = Dataset.from_dict({"text": [val_text]})
print("Raw Validation Dataset created:")
print(eval_dataset_raw)

In [ ]:
def tokenize_and_chunk(examples):
    tokenized_output = tokenizer(examples["text"], truncation=False, add_special_tokens=False) # Avoid adding BOS/EOS here if tokenizer does it automatically

    concatenated_examples = {k: sum(tokenized_output[k], []) for k in tokenized_output.keys()}
    total_length = len(concatenated_examples[list(tokenized_output.keys())[0]])

    if total_length >= max_seq_length:
        total_length = (total_length // max_seq_length) * max_seq_length
    else:
         # Handle case where total tokens < max_seq_length if needed, maybe pad later?
         # For CPT, dropping the remainder is common. If total < max_seq, result will be empty.
         print(f"Warning: Total token length ({total_length}) is less than max_seq_length ({max_seq_length}). No chunks generated.")
         return {"input_ids": [], "attention_mask": [], "labels": []} # Return empty


    result = {
        k: [t[i : i + max_seq_length] for i in range(0, total_length, max_seq_length)]
        for k, t in concatenated_examples.items()
    }
    result["labels"] = result["input_ids"].copy()
    return result

num_proc_tok = 4


# --- Tokenize Datasets ---
print(f"Tokenizing combined training dataset (max_seq_length = {max_seq_length})...")
tokenized_train_dataset = train_dataset_raw.map(
    tokenize_and_chunk,
    batched=True,
    num_proc=num_proc_tok,
    remove_columns=["text"],
)
print("Tokenized Training Dataset:")
print(tokenized_train_dataset)


print(f"Tokenizing validation dataset (max_seq_length = {max_seq_length})...")
tokenized_eval_dataset = eval_dataset_raw.map(
    tokenize_and_chunk,
    batched=True,
    num_proc=num_proc_tok,
    remove_columns=["text"],
)
print("Tokenized Validation Dataset:")
print(tokenized_eval_dataset)

# --- Validation Set Size Check ---
num_train_rows = len(tokenized_train_dataset)
num_eval_rows = len(tokenized_eval_dataset)
print(f"\nNumber of training sequences (rows): {num_train_rows}")
print(f"Number of validation sequences (rows): {num_eval_rows}")

if num_train_rows > 0 and num_eval_rows > 0 :
    eval_percentage = (num_eval_rows / (num_train_rows + num_eval_rows)) * 100
    print(f"Validation set size is approx. {eval_percentage:.2f}% of the total tokenized sequences.")
elif num_train_rows == 0:
     print("Warning: No training sequences generated after tokenization. Check data and max_seq_length.")
elif num_eval_rows == 0:
     print("Warning: No validation sequences generated after tokenization. Check data and max_seq_length.")

In [ ]:
from transformers import TrainingArguments, DataCollatorForLanguageModeling
from unsloth import is_bfloat16_supported
from unsloth import UnslothTrainer, UnslothTrainingArguments

data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

trainer = UnslothTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = tokenized_train_dataset,
    data_collator = data_collator,
    max_seq_length = max_seq_length,
    eval_dataset = tokenized_eval_dataset# Use the defined value

    args = UnslothTrainingArguments(
        per_device_train_batch_size = 4,         # Must be 1 for 16GB T4 CPT likely
        gradient_accumulation_steps = 16,        # Effective batch size = 16. Can increase (eg 32) if VRAM allows & want stability.
        warmup_ratio = 0.1,                      # Warmup over 10% of training
        # Choose ONE of max_steps or num_train_epochs
        # Option 1: Max Steps (easier control for CPT)
        max_steps = 100,                      # Adjust based on dataset size & time. Start smaller (eg 500) to test for OOM.
        # Option 2: Epochs (use if you want full passes)
        # num_train_epochs = 1,                # Example: Train for 1 full epoch

        # Lower learning rates for CPT
        learning_rate = 1e-5,                    # Lowered main LR
        embedding_learning_rate = 1e-6,          # Lowered embedding LR (10x ratio)

        # T4 only supports FP16
        fp16 = True,
        bf16 = False,

        logging_steps = 1,    
        save_total_limit = 2,
        optim = "paged_adamw_8bit",            # Most memory efficient AdamW variant
        weight_decay = 0.01,                     # Regularization, recommended over 0.00
        lr_scheduler_type = "cosine",
        evaluation_strategy = "steps",
        eval_steps = 50,
        save_strategy = "steps",
        save_steps = 50,
        save_total_limit = 2,
        output_dir = "outputs_math_cpt_t4",       # Specific output dir name
        report_to = "none",                      # Change if using WandB/Tensorboard
    ),
)

In [ ]:
trainer_stats = trainer.train()

In [ ]:
print("Training finished. Saving final adapter model...")

# Define the output directory in the Kaggle working directory
final_adapter_path = "/kaggle/working/final_math_adapter"

# Save the LoRA adapters
model.save_pretrained(final_adapter_path)

# Optionally, save the tokenizer as well (recommended)
tokenizer.save_pretrained(final_adapter_path)

print(f"Adapter model saved to: {final_adapter_path}")

In [ ]:
print("\nReloading base model and applying the saved adapter for inference...")

from unsloth import FastLanguageModel
import torch

# Make sure these parameters match how you loaded the model initially
dtype = None # Or Float16, BFloat16
load_in_4bit = True
max_seq_length = 2048 

model_inf, tokenizer_inf = FastLanguageModel.from_pretrained(
    model_name = final_adapter_path, # <--- Load the saved adapter directory
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
)

model_inf.eval()

print("Model ready for inference with the fine-tuned adapter.")

In [ ]:
# Prompt that sets up the context for an equation
prompt = "What are Euler’s and Lagrange’s central configurations for three bodies?"

inputs = tokenizer_inf([prompt], return_tensors="pt").to("cuda")
with torch.no_grad():
    outputs = model_inf.generate(**inputs, max_new_tokens=2048, use_cache=True, pad_token_id=tokenizer_inf.eos_token_id)
    generated_text = tokenizer_inf.batch_decode(outputs)[0]
    print("Model Response:\n", generated_text)

In [ ]:
# Prompt that sets up the context for an equation
prompt = "Implement SymPy code to verify whether three given masses at specific positions form a central configuration."

inputs = tokenizer_inf([prompt], return_tensors="pt").to("cuda")
with torch.no_grad():
    outputs = model_inf.generate(**inputs, max_new_tokens=2048, use_cache=True, pad_token_id=tokenizer_inf.eos_token_id)
    generated_text = tokenizer_inf.batch_decode(outputs)[0]
    print("Model Response:\n", generated_text)

In [ ]:
# Prompt that sets up the context for an equation
prompt = "Singularities in an n‑body system occur when two or more bodies come extremely close to each other, causing the gravitational forces (and hence accelerations) to approach infinity. Please provide a clear LaTeX explanation showing how such singularities (specifically collision singularities) appear in the equations of motion derived from Newton’s laws. Then, write complete Python code using the SymPy library to simulate a near‑collision scenario for three bodies in 2D space. Use explicit initial conditions for the positions and velocities. In your code, include comments on the numerical challenges encountered during the simulation (such as instability due to large forces near collisions), and describe any potential strategies (such as regularization techniques or adaptive time‑stepping) that could be used to mitigate these issues. Ensure that the code is self-contained and runnable."

inputs = tokenizer_inf([prompt], return_tensors="pt").to("cuda")
with torch.no_grad():
    outputs = model_inf.generate(**inputs, max_new_tokens=2048, use_cache=True, pad_token_id=tokenizer_inf.eos_token_id)
    generated_text = tokenizer_inf.batch_decode(outputs)[0]
    print("Model Response:\n", generated_text)